# 14 — Dashboard thị trường hằng ngày

**Sản phẩm 1.** Ghép ba notebook trước thành một báo cáo một trang, chạy được
mỗi chiều sau phiên ATC và xuất ra file HTML mở bằng trình duyệt.

Báo cáo trả lời năm câu, theo thứ tự người đọc cần chúng:

1. Chỉ số đóng cửa bao nhiêu, tăng giảm thế nào?
2. Thanh khoản có bình thường không?
3. Thị trường **rộng** hay hẹp — bao nhiêu mã thực sự tăng?
4. Tiền chảy vào ngành nào, ra khỏi ngành nào?
5. Mã nào đáng nhìn?

Toàn bộ chạy trong khoảng **8 lời gọi API**.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, hom_nay, lui_ngay, ty_dong
from finlens_examples.charts import GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

## 1 · Thu thập

Gom hết dữ liệu lên trước, ở một chỗ. Mỗi lời gọi có chú thích nói nó phục vụ
phần nào của báo cáo — sáu tháng sau bạn sẽ cần chú thích đó.

In [2]:
CHI_SO = ["VNINDEX", "VN30", "HNXINDEX", "UPINDEX"]
BAT_DAU = lui_ngay(HOM_NAY, thang=3)

# Danh mục mã + ngành — nền tảng của mọi phần sau
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
MA_HOSE = danh_muc["symbol"].tolist()
cap2 = client.meta.sectors(level=2)

# Phần 1 & 2: chỉ số và thanh khoản
chi_so = client.eod.index.ohlcv(CHI_SO, start=BAT_DAU)

# Phần 3 & 5: giá toàn sàn (breadth, top tăng/giảm)
gia = client.eod.stock.ohlcv(MA_HOSE, start=lui_ngay(HOM_NAY, thang=6))

# Phần 4: dòng tiền ngoại theo ngành và theo mã
dong_nganh = client.eod.sector.investor.flow(
    cap2["icb"].tolist(), icb_level=2, group="foreign", start=lui_ngay(HOM_NAY, ngay=30)
)
dong_ma = client.eod.stock.investor.flow(MA_HOSE, group="foreign", start=lui_ngay(HOM_NAY, ngay=10))

PHIEN = gia["date"].max()
print(f"Phiên báo cáo: {PHIEN:%d/%m/%Y}")
print(f"Mốc nước dữ liệu giá: {gia.attrs['finlens']['as_of']}")

C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


Phiên báo cáo: 11/08/2026
Mốc nước dữ liệu giá: 2026-08-11T00:00:00+07:00


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


⚠️ `PHIEN` lấy từ **dữ liệu**, không phải từ `HOM_NAY`. Chạy báo cáo vào thứ
Bảy, hay vào 9h sáng trước khi số liệu hôm trước kịp về, `HOM_NAY` sẽ trỏ vào
một phiên không tồn tại và mọi bộ lọc `date == HOM_NAY` trả về bảng rỗng.

## 2 · Phần 1–2: chỉ số và thanh khoản

In [3]:
def tom_tat_chi_so(df: pd.DataFrame) -> pd.DataFrame:
    """Đóng cửa, thay đổi phiên, và thay đổi từ đầu kỳ cho từng chỉ số."""
    df = df.sort_values(["symbol", "date"])
    g = df.groupby("symbol", observed=True)
    return pd.DataFrame(
        {
            "dong_cua": g["close"].last(),
            "thay_doi_phien_pct": g["close"].apply(lambda s: (s.iloc[-1] / s.iloc[-2] - 1) * 100),
            "thay_doi_3thang_pct": g["close"].apply(lambda s: (s.iloc[-1] / s.iloc[0] - 1) * 100),
            "gtgd_ty": g["volume"].last() / 1e6,  # chỉ số: volume là cổ phiếu
        }
    ).round(2)


tt_chi_so = tom_tat_chi_so(chi_so)
tt_chi_so

,dong_cua,thay_doi_phien_pct,thay_doi_3thang_pct,gtgd_ty
symbol,,,,
HNXINDEX,290.91,1.12,14.25,44.70
UPINDEX,127.18,-0.11,0.71,17.61
VN30,1922.53,-0.14,-5.92,217.02
VNINDEX,1773.41,-0.19,-6.58,553.92


In [4]:
# Thanh khoản toàn sàn: giá (nghìn VND) × khối lượng × 1.000 = VND
gia_gtgd = gia.assign(gtgd=gia["close"] * gia["volume"] * 1_000)
gtgd_phien = gia_gtgd.groupby("date", observed=True)["gtgd"].sum() / 1e9

gtgd_hom_nay = gtgd_phien.loc[PHIEN]
gtgd_bq_20 = gtgd_phien.tail(21).iloc[:-1].mean()

print(f"GTGD phiên này:        {gtgd_hom_nay:>10,.0f} tỷ đồng")
print(f"Bình quân 20 phiên:    {gtgd_bq_20:>10,.0f} tỷ đồng")
print(f"So với bình quân:      {gtgd_hom_nay / gtgd_bq_20 - 1:>+10.1%}")

GTGD phiên này:            14,000 tỷ đồng
Bình quân 20 phiên:        14,684 tỷ đồng
So với bình quân:           -4.7%


## 3 · Phần 3: độ rộng

In [5]:
co_ma = gia.sort_values(["symbol", "date"]).finlens.sma(50)

phien_nay = co_ma[co_ma["date"] == PHIEN].dropna(subset=["sma_50"])
pct_tren_ma50 = (phien_nay["close"] > phien_nay["sma_50"]).mean() * 100

thay_doi_phien = (
    co_ma.sort_values(["symbol", "date"])
    .groupby("symbol", observed=True)["close"]
    .pct_change()
    .rename("ls")
)
hom_nay_ls = co_ma.assign(ls=thay_doi_phien).query("date == @PHIEN").dropna(subset=["ls"])

so_tang = int((hom_nay_ls["ls"] > 0).sum())
so_giam = int((hom_nay_ls["ls"] < 0).sum())
so_dung = int((hom_nay_ls["ls"] == 0).sum())

print(f"Tăng {so_tang} · Giảm {so_giam} · Đứng giá {so_dung}")
print(f"{pct_tren_ma50:.1f}% số mã nằm trên MA50")

Tăng 162 · Giảm 155 · Đứng giá 86
34.0% số mã nằm trên MA50


D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\IPython\core\interactiveshell.py:3715: DataQualityWarning: `SMA` cần ít nhất 50 dòng mỗi nhóm nhưng 1 nhóm (`symbol`) ngắn hơn thế: SVI. Chúng ra TOÀN `NaN` — TA-Lib không báo lỗi cho trường hợp này.
  if await self.run_code(code, result, async_=asy):


## 4 · Phần 4: dòng tiền ngoại

In [6]:
ngoai_phien = dong_ma[dong_ma["date"] == dong_ma["date"].max()]
ngoai_rong = ngoai_phien["net_value"].sum() / 1e9

print(f"Khối ngoại toàn sàn: {ngoai_rong:+,.0f} tỷ đồng")
print(f"(số liệu dòng tiền tới phiên {dong_ma['date'].max():%d/%m/%Y})")

nganh_ngoai = (
    dong_nganh.groupby("icb_name", observed=True)["net_value"].sum().div(1e9).round(0).reset_index()
)

Khối ngoại toàn sàn: -758 tỷ đồng
(số liệu dòng tiền tới phiên 11/08/2026)


⚠️ Dòng tiền nhà đầu tư thường về **chậm hơn giá một phiên**. Báo cáo phải nói
rõ hai mốc thời gian thay vì gộp chúng làm một — nếu không, người đọc sẽ tưởng
con số khối ngoại là của phiên hôm nay.

## 5 · Phần 5: mã đáng nhìn

Ba bảng, ba tiêu chí khác nhau — vì "đáng nhìn" không có một định nghĩa duy nhất.

In [7]:
# Lọc thanh khoản trước, luôn luôn
gtgd_ma = (
    gia_gtgd[gia_gtgd["date"] > PHIEN - pd.Timedelta(days=30)]
    .groupby("symbol", observed=True)["gtgd"]
    .mean()
)
MA_LONG = gtgd_ma[gtgd_ma >= 10e9].index  # ≥ 10 tỷ/phiên

bien_dong = (
    hom_nay_ls[hom_nay_ls["symbol"].isin(MA_LONG)]
    .assign(thay_doi_pct=lambda d: d["ls"] * 100)
    .merge(danh_muc[["symbol", "icb_name2"]], on="symbol")
)

top_tang = bien_dong.nlargest(10, "thay_doi_pct")
top_giam = bien_dong.nsmallest(10, "thay_doi_pct")

# Đột biến khối lượng: khối lượng phiên này so với bình quân 20 phiên của chính nó
kl_bq = (
    gia[gia["date"] > PHIEN - pd.Timedelta(days=30)]
    .groupby("symbol", observed=True)["volume"]
    .mean()
    .rename("kl_bq20")
)
dot_bien = (
    gia[gia["date"] == PHIEN]
    .merge(kl_bq, on="symbol")
    .query("symbol in @MA_LONG")
    .assign(so_lan=lambda d: d["volume"] / d["kl_bq20"])
    .nlargest(10, "so_lan")
    .merge(bien_dong[["symbol", "thay_doi_pct"]], on="symbol", how="left")
)

print(f"Vũ trụ sau lọc thanh khoản: {len(MA_LONG)}/{gia['symbol'].nunique()} mã")
dot_bien[["symbol", "close", "volume", "kl_bq20", "so_lan", "thay_doi_pct"]].round(2)

Vũ trụ sau lọc thanh khoản: 114/404 mã

,symbol,close,volume,kl_bq20,so_lan,thay_doi_pct
0,OCB,10.85,7715800.0,2556204.55,3.02,1.40
1,HHP,16.50,3204500.0,1147890.91,2.79,0.61
2,BCM,41.15,1897500.0,692304.55,2.74,-0.36
3,VTP,54.50,1450100.0,546886.36,2.65,2.06
4,FRT,146.50,1428500.0,542345.45,2.63,6.93
5,GEE,74.30,3288900.0,1249040.91,2.63,6.91
6,PVT,19.70,7670700.0,3476795.45,2.21,1.55
7,DHC,35.20,786100.0,392609.09,2.00,0.14
8,ORS,14.45,9520800.0,4848100.00,1.96,1.76
9,VPI,60.00,2684400.0,1615740.91,1.66,-2.76


## 6 · Dựng báo cáo

Ba biểu đồ, đủ để trả lời năm câu hỏi. Thêm biểu đồ thứ tư là bắt người đọc
phải chọn nhìn cái nào — mà chọn giúp họ chính là việc của báo cáo.

In [8]:
fig_nganh = bar_ngang(
    nganh_ngoai,
    nhan="icb_name",
    gia_tri="net_value",
    tieu_de="Khối ngoại theo ngành — 30 phiên",
    phu_de="Cộng dồn giá trị mua/bán ròng",
    nhan_x="tỷ đồng",
    dinh_dang_nhan="{:+,.0f}",
)

fig_tang_giam = bar_ngang(
    pd.concat([top_tang, top_giam]),
    nhan="symbol",
    gia_tri="thay_doi_pct",
    tieu_de=f"Top tăng / giảm phiên {PHIEN:%d/%m}",
    phu_de="Chỉ tính mã có GTGD bình quân ≥ 10 tỷ/phiên",
    nhan_x="%",
    dinh_dang_nhan="{:+.1f}%",
)

fig_dot_bien = bar_ngang(
    dot_bien,
    nhan="symbol",
    gia_tri="so_lan",
    tieu_de="Đột biến khối lượng",
    phu_de="Khối lượng phiên này chia bình quân 20 phiên của chính mã đó",
    nhan_x="lần",
    dinh_dang_nhan="{:.1f}×",
)

fig_tang_giam

### Ô số liệu (stat tile)

Năm con số quan trọng nhất **không nên là biểu đồ**. Một biểu đồ một điểm dữ
liệu là biểu đồ thừa; con số lớn đọc nhanh hơn và chính xác hơn.

In [9]:
def o_so_lieu(nhan: str, gia_tri: str, phu: str = "", dau: float | None = None) -> str:
    """Một ô số liệu HTML. `dau` quyết định màu: dương aqua, âm đỏ, None trung tính."""
    mau = "#0b0b0b" if dau is None else (TANG if dau >= 0 else GIAM)
    return f"""
    <div class="o">
      <div class="nhan">{nhan}</div>
      <div class="gia-tri" style="color:{mau}">{gia_tri}</div>
      <div class="phu">{phu}</div>
    </div>"""


vni = tt_chi_so.loc["VNINDEX"]

cac_o = "".join(
    [
        o_so_lieu(
            "VNINDEX",
            f"{vni['dong_cua']:,.2f}",
            f"{vni['thay_doi_phien_pct']:+.2f}% phiên",
            vni["thay_doi_phien_pct"],
        ),
        o_so_lieu(
            "Thanh khoản HOSE",
            f"{gtgd_hom_nay:,.0f} tỷ",
            f"{gtgd_hom_nay / gtgd_bq_20 - 1:+.0%} so với BQ 20 phiên",
            gtgd_hom_nay - gtgd_bq_20,
        ),
        o_so_lieu(
            "Độ rộng",
            f"{so_tang} ↑ / {so_giam} ↓",
            f"{pct_tren_ma50:.0f}% số mã trên MA50",
            so_tang - so_giam,
        ),
        o_so_lieu(
            "Khối ngoại",
            f"{ngoai_rong:+,.0f} tỷ",
            f"phiên {dong_ma['date'].max():%d/%m}",
            ngoai_rong,
        ),
        o_so_lieu(
            "VN30",
            f"{tt_chi_so.loc['VN30', 'dong_cua']:,.2f}",
            f"{tt_chi_so.loc['VN30', 'thay_doi_phien_pct']:+.2f}% phiên",
            tt_chi_so.loc["VN30", "thay_doi_phien_pct"],
        ),
    ]
)

from IPython.display import HTML

CSS = """
<style>
.bang-o { display:flex; gap:12px; flex-wrap:wrap; font-family:system-ui,-apple-system,"Segoe UI",sans-serif; margin:8px 0 20px; }
.o { flex:1 1 170px; background:#fcfcfb; border:1px solid rgba(11,11,11,.10); border-radius:10px; padding:14px 16px; }
.nhan { font-size:12px; color:#52514e; letter-spacing:.02em; }
.gia-tri { font-size:26px; font-weight:600; margin:4px 0 2px; }
.phu { font-size:12px; color:#898781; }
</style>
"""

HTML(CSS + f'<div class="bang-o">{cac_o}</div>')

## 7 · Xuất ra file HTML

`include_plotlyjs=True` ở biểu đồ **đầu tiên** và `False` ở các biểu đồ sau:
thư viện plotly.js chỉ nhúng một lần thay vì ba lần. Bỏ qua chi tiết này thì
file phình lên gấp ba mà nội dung không đổi.

Nhúng thẳng (thay vì `include_plotlyjs="cdn"`) để file mở được **không cần
mạng** — báo cáo gửi qua email hay mở trên máy không có internet vẫn hiện.

In [10]:
def xuat_bao_cao(duong_dan: Path) -> Path:
    """Ghi báo cáo một trang thành file HTML độc lập."""
    khoi = [
        fig_tang_giam.to_html(full_html=False, include_plotlyjs=True),
        fig_nganh.to_html(full_html=False, include_plotlyjs=False),
        fig_dot_bien.to_html(full_html=False, include_plotlyjs=False),
    ]
    html = f"""<!doctype html>
<html lang="vi"><head><meta charset="utf-8">
<title>Báo cáo thị trường {PHIEN:%d/%m/%Y}</title>
{CSS}
<style>
  body {{ background:#f9f9f7; color:#0b0b0b; margin:0; padding:28px 32px 48px;
         font-family:system-ui,-apple-system,"Segoe UI",sans-serif; }}
  h1 {{ font-size:22px; margin:0 0 2px; }}
  .moc {{ color:#898781; font-size:13px; margin-bottom:18px; }}
  .bieu-do {{ background:#fcfcfb; border:1px solid rgba(11,11,11,.10);
              border-radius:10px; padding:8px; margin-bottom:16px; }}
  footer {{ color:#898781; font-size:12px; margin-top:24px;
            border-top:1px solid #e1e0d9; padding-top:12px; }}
</style></head>
<body>
  <h1>Báo cáo thị trường — phiên {PHIEN:%d/%m/%Y}</h1>
  <div class="moc">Giá tới {gia.attrs["finlens"]["as_of"]} ·
       dòng tiền tới phiên {dong_ma["date"].max():%d/%m/%Y} ·
       {len(MA_LONG)} mã qua bộ lọc thanh khoản</div>
  <div class="bang-o">{cac_o}</div>
  {"".join(f'<div class="bieu-do">{k}</div>' for k in khoi)}
  <footer>Nguồn: FinLens · sinh tự động từ notebook <code>14_dashboard_hang_ngay.ipynb</code></footer>
</body></html>"""
    duong_dan.write_text(html, encoding="utf-8")
    return duong_dan


tep = xuat_bao_cao(THU_MUC_RA / f"bao_cao_thi_truong_{PHIEN:%Y%m%d}.html")
print(f"Đã ghi: {tep}")
print(f"Kích thước: {tep.stat().st_size / 1024:,.0f} KB")

Đã ghi: D:\finlens\finlens-python\finlens-python-example\output\bao_cao_thi_truong_20260811.html
Kích thước: 4,754 KB


## 8 · Chạy tự động mỗi ngày

Notebook này chạy được bằng dòng lệnh, không cần mở Jupyter:

```bash
jupyter nbconvert --to notebook --execute --inplace notebooks/14_dashboard_hang_ngay.ipynb
```

Đặt vào Task Scheduler (Windows) hoặc cron lúc 15h30 mỗi ngày làm việc là có
báo cáo tự động. Ba điều cần thêm khi chạy không người trông:

1. **`on_error="raise"`** ở mọi lời gọi — im lặng bỏ qua một mã hỏng là đúng
   khi bạn đang khám phá, và là sai khi không ai nhìn.
2. **Bắt `finlens.QuotaError`** và dừng có kiểm soát thay vì để traceback.
3. **Kiểm `PHIEN`** — nếu nó không phải hôm nay thì dữ liệu chưa về; báo cáo
   vẫn đúng nhưng phải nói rõ nó là của phiên nào.

In [11]:
def kiem_tra_truoc_khi_chay() -> None:
    """Ba khẳng định nên chạy đầu mỗi lần chạy tự động."""
    han = client.limits()
    assert han["requests_remaining"] > 50, f"Sắp hết hạn mức: còn {han['requests_remaining']}"
    assert not gia.attrs["finlens"]["truncated"], "Dữ liệu giá bị cắt vì max_rows"
    if PHIEN.date() != HOM_NAY:
        print(f"⚠️  Phiên gần nhất là {PHIEN:%d/%m}, không phải {HOM_NAY:%d/%m} — dữ liệu chưa về hoặc hôm nay nghỉ.")
    else:
        print("✓ Dữ liệu là của phiên hôm nay.")


kiem_tra_truoc_khi_chay()

✓ Dữ liệu là của phiên hôm nay.


## Tổng kết

Báo cáo này dùng **8 lời gọi API** và chạy trong vài chục giây. Cấu trúc đáng
giữ lại cho bất kỳ báo cáo nào bạn viết sau:

| Bước | Vì sao tách riêng |
|---|---|
| Thu thập gọn một chỗ | đếm được số request, và cache dùng lại được |
| `PHIEN` lấy từ dữ liệu | thứ Bảy hay lễ không làm báo cáo rỗng |
| Lọc thanh khoản trước mọi xếp hạng | nếu không, bảng xếp hạng nhiễu |
| Hai mốc thời gian nói rõ | giá và dòng tiền không cùng độ trễ |
| `plotlyjs` nhúng một lần | file nhỏ gấp ba, mở được không cần mạng |

---

**Track 1 hết.** Tiếp theo là Track 2:
[`21_bao_cao_tai_chinh.ipynb`](../02-phan-tich-co-ban/21_bao_cao_tai_chinh.ipynb) — báo cáo tài chính
dạng long, cây chỉ tiêu, và bốn loại hình doanh nghiệp.